# v1a.1 post-training calibration
Validation only. Do not use the test split until the final configuration is frozen.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
PROJECT = '/content/v1a1_dual_kidney_pipeline_calibrated'
MANIFEST = '/content/v1a1_work/manifest/v1a1_dual_manifest.csv'
RUN_DRIVE = '/content/drive/MyDrive/medsam-kits23/v1a1_results/run'
CAL = '/content/v1a1_work/calibration'
MEDSAM_CKPT = '/content/drive/MyDrive/medsam_vit_b.pth'
%cd {PROJECT}


## Export candidate validation predictions


In [ ]:
!mkdir -p {CAL}/epoch5 {CAL}/epoch6 {CAL}/epoch7
!python scripts/predict_dual.py --manifest {MANIFEST} --checkpoint {RUN_DRIVE}/best_continuous.pt --output-csv {CAL}/epoch6/val_base_predictions.csv --split val --weights ema --batch-size 32 --device cuda
!python scripts/predict_dual.py --manifest {MANIFEST} --checkpoint {RUN_DRIVE}/best_prompt.pt --output-csv {CAL}/epoch7/val_base_predictions.csv --split val --weights ema --batch-size 32 --device cuda
!python scripts/predict_dual.py --manifest {MANIFEST} --checkpoint {RUN_DRIVE}/checkpoints/epoch_005.pt --output-csv {CAL}/epoch5/val_base_predictions.csv --split val --weights ema --batch-size 32 --device cuda


## Threshold + smoothing calibration
Run this block for epoch 5/6/7.


In [ ]:
for EPOCH in ('epoch5','epoch6','epoch7'):
    !python scripts/calibrate_presence_smoothing.py --predictions {CAL}/{EPOCH}/val_base_predictions.csv --output-dir {CAL}/{EPOCH}/presence_smoothing --left-thresholds 0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55 --right-thresholds 0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55 --max-gaps 0,1,2 --expansion-scale 1.15 --min-overall-recall 0.95 --min-side-recall 0.90


## Prompt expansion sweep


In [ ]:
for EPOCH in ('epoch5','epoch6','epoch7'):
    !python scripts/sweep_expansion_calibrated.py --predictions {CAL}/{EPOCH}/presence_smoothing/recommended_predictions.csv --calibration-json {CAL}/{EPOCH}/presence_smoothing/recommended_calibration.json --output-dir {CAL}/{EPOCH}/expansion --scales 1.10,1.15,1.20,1.25


## Set up frozen MedSAM


In [ ]:
!python scripts/setup_medsam.py --repo-dir /content/v1a1_work/MedSAM --checkpoint-source {MEDSAM_CKPT} --checkpoint-dest /content/v1a1_work/medsam_weights/medsam_vit_b.pth


## MedSAM pilot (100 validation slices)


In [ ]:
for EPOCH in ('epoch5','epoch6','epoch7'):
    !python scripts/medsam_validation_sweep.py --predictions {CAL}/{EPOCH}/presence_smoothing/recommended_predictions.csv --calibration-json {CAL}/{EPOCH}/presence_smoothing/recommended_calibration.json --medsam-repo /content/v1a1_work/MedSAM --checkpoint /content/v1a1_work/medsam_weights/medsam_vit_b.pth --output-dir {CAL}/{EPOCH}/medsam_pilot --scales 1.10,1.15,1.20,1.25 --max-samples 100 --device cuda


After the pilot succeeds, repeat without `--max-samples 100`, writing to `medsam_full`. Then compare candidates.


In [ ]:
!python scripts/compare_calibrated_candidates.py --candidate epoch5={CAL}/epoch5/medsam_full/medsam_validation_summary.json --candidate epoch6={CAL}/epoch6/medsam_full/medsam_validation_summary.json --candidate epoch7={CAL}/epoch7/medsam_full/medsam_validation_summary.json --output-dir {CAL}/comparison
